# Understanding the Original Study


# Data Preparation

### Access Google Cluster Data

So we have 2011's Google cluster Data to test locally with ~42GB worth of trace data
As well as a 2.4 TiB Google Cluster Data sample I'll try to query with BigQuery if I get to it.

in other words this is a lot of data to go through on a personal computer and may not reccomend if your hardware doesn't have the space for it.
### Extract Relevant Data Using AGOCS

### Filter TAsk by Node Affinity

In [ ]:
import pandas as pd
import numpy as np

# Load task constraint data
task_constraints = pd.read_csv('task_constraints.csv')

# Filter for tasks with specific node requirements
# Constraint name 3 typically represents node affinity in GCD
node_affinity_tasks = task_constraints[
    task_constraints['constraint_name'] == 3  # Adjust based on actual constraint type
]

# Filter for:
# 1. Tasks with single node requirement
single_node_tasks = node_affinity_tasks[
    node_affinity_tasks['machine_id'].nunique() == 1
].groupby('job_id', 'task_index')

# 2. Tasks with <1000 node options
limited_node_tasks = node_affinity_tasks.groupby(['job_id', 'task_index']).filter(
    lambda x: x['machine_id'].nunique() < 1000
)

print(f"Single node tasks: {len(single_node_tasks)}")
print(f"Limited node tasks (<1000): {len(limited_node_tasks)}")

### One-Hot Encode Constraint Operators

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Extract unique constraint operators
# Compact constraint operators (remove duplicates, normalize)
def compact_constraints(df):
    # Group by task and create compact representation
    constraint_features = df.groupby(['job_id', 'task_index']).agg({
        'constraint_name': list,
        'comparison_operator': list,
        'value': list
    }).reset_index()
    
    # Create feature strings for one-hot encoding
    constraint_features['constraint_string'] = constraint_features.apply(
        lambda row: '_'.join([
            f"{name}_{op}_{val}" 
            for name, op, val in zip(
                row['constraint_name'], 
                row['comparison_operator'], 
                row['value']
            )
        ]), axis=1
    )
    
    return constraint_features

# Apply one-hot encoding
constraints_compact = compact_constraints(task_constraints)
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
X_encoded = encoder.fit_transform(
    constraints_compact[['constraint_string']]
)

# Create feature names
feature_names = encoder.get_feature_names_out()
X_df = pd.DataFrame(X_encoded, columns=feature_names)

# Creating Comparable Train/Test Splits

### Match the Original Study's Data Split

In [ ]:
from sklearn.model_selection import train_test_split

# Ensure reproducibility
np.random.seed(42)

# Create target variable (node_id or binary single/multi-node)
y = constraints_compact['target_node_id']  # Adjust based on your task

# Split ratio - typical is 80/20 or 70/30
# Paper doesn't specify, so test both
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintain class distribution
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Feature dimensions: {X_train.shape[1]}")

### Separate Single-Node Vs Multi-Node Tasks

In [ ]:
# For more detailed analysis matching the paper
def create_task_type_splits(X, y, task_metadata):
    """
    Split data by task type for detailed evaluation
    """
    # Identify single-node tasks
    single_node_mask = task_metadata['num_candidate_nodes'] == 1
    multi_node_mask = task_metadata['num_candidate_nodes'] > 1
    
    return {
        'single_node': {
            'X': X[single_node_mask],
            'y': y[single_node_mask]
        },
        'multi_node': {
            'X': X[multi_node_mask],
            'y': y[multi_node_mask]
        }
    }

splits = create_task_type_splits(X_test, y_test, test_metadata)

# Implementing the Baseline Ensemble 

### Replicate the Ensemble Voting Classifier

In [ ]:
from sklearn.ensemble import (
    VotingClassifier, AdaBoostClassifier, 
    BaggingClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Define base classifiers with tuning
classifiers = [
    ('ann', MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42
    )),
    ('knn', KNeighborsClassifier(
        n_neighbors=5,
        weights='distance'
    )),
    ('dt', DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    )),
    ('nb', GaussianNB()),
    ('ridge', RidgeClassifier(
        random_state=42
    )),
    ('adaboost', AdaBoostClassifier(
        n_estimators=50,
        random_state=42
    )),
    ('bagging', BaggingClassifier(
        n_estimators=50,
        random_state=42
    ))
]

# Create ensemble voting classifier
ensemble = VotingClassifier(
    estimators=classifiers,
    voting='soft',  # Use probability-based voting
    n_jobs=-1
)

# Train the ensemble
print("Training ensemble classifier...")
ensemble.fit(X_train, y_train)

# Evaluate
y_pred_ensemble = ensemble.predict(X_test)
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)
ensemble_f1 = f1_score(y_test, y_pred_ensemble, average='weighted')

print(f"Ensemble Accuracy: {ensemble_accuracy:.4f}")
print(f"Ensemble F1-Score: {ensemble_f1:.4f}")

### Now it's time for our addition
# Implementing XGBoost

### Basic XGBoost Implementation 

In [ ]:
import xgboost as xgb
from sklearn.metrics import confusion_matrix
import time

# Create XGBoost classifier
xgb_classifier = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',  # or 'binary:logistic' for binary classification
    num_class=len(np.unique(y)),  # Number of target classes
    random_state=42,
    tree_method='hist',  # Faster training
    n_jobs=-1
)

# Measure training time
start_time = time.time()
xgb_classifier.fit(X_train, y_train)
training_time = time.time() - start_time

# Measure prediction time
start_time = time.time()
y_pred_xgb = xgb_classifier.predict(X_test)
prediction_time = time.time() - start_time

# Evaluate
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")
print(f"XGBoost F1-Score: {xgb_f1:.4f}")
print(f"Training Time: {training_time:.2f}s")
print(f"Prediction Time: {prediction_time:.4f}s")

### Hyper tune

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# Grid search with cross-validation
grid_search = GridSearchCV(
    xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(np.unique(y)),
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Use best model
best_xgb = grid_search.best_estimator_

# Comparative Evaluation

### Create Comprehensive Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def comprehensive_comparison(models_dict, X_test, y_test, test_metadata):
    """
    Compare models across multiple dimensions
    """
    results = []
    
    for model_name, model in models_dict.items():
        # Overall metrics
        y_pred = model.predict(X_test)
        
        metrics = {
            'Model': model_name,
            'Overall_Accuracy': accuracy_score(y_test, y_pred),
            'Overall_F1': f1_score(y_test, y_pred, average='weighted'),
            'Overall_Misclassification': 1 - accuracy_score(y_test, y_pred)
        }
        
        # Single-node task metrics (matching paper's 1.5-1.8%)
        single_node_mask = test_metadata['num_candidate_nodes'] == 1
        if single_node_mask.any():
            y_test_single = y_test[single_node_mask]
            y_pred_single = y_pred[single_node_mask]
            
            metrics.update({
                'SingleNode_Accuracy': accuracy_score(y_test_single, y_pred_single),
                'SingleNode_Misclassification': 1 - accuracy_score(y_test_single, y_pred_single),
                'SingleNode_F1': f1_score(y_test_single, y_pred_single, average='weighted')
            })
        
        # Timing
        start = time.time()
        _ = model.predict(X_test[:1000])  # Sample for speed test
        inference_time = (time.time() - start) / 1000
        metrics['Inference_Time_per_Sample_ms'] = inference_time * 1000
        
        results.append(metrics)
    
    return pd.DataFrame(results)

# Compare models
models = {
    'Ensemble': ensemble,
    'XGBoost': best_xgb,
    'XGBoost_Default': xgb_classifier
}

comparison_df = comprehensive_comparison(models, X_test, y_test, test_metadata)
print(comparison_df.to_string(index=False))

### Statistical Significance Testing

In [ ]:
from scipy import stats

def statistical_comparison(model1_preds, model2_preds, y_test):
    """
    Test if performance difference is statistically significant
    """
    # McNemar's test for paired binary outcomes
    correct_1 = (model1_preds == y_test).astype(int)
    correct_2 = (model2_preds == y_test).astype(int)
    
    # Create contingency table
    n01 = np.sum((correct_1 == 0) & (correct_2 == 1))
    n10 = np.sum((correct_1 == 1) & (correct_2 == 0))
    
    statistic = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p_value = 1 - stats.chi2.cdf(statistic, df=1)
    
    return {
        'statistic': statistic,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

# Test XGBoost vs Ensemble
y_pred_ensemble = ensemble.predict(X_test)
y_pred_xgb = best_xgb.predict(X_test)

sig_test = statistical_comparison(y_pred_ensemble, y_pred_xgb, y_test)
print(f"Statistical Test Results:")
print(f"P-value: {sig_test['p_value']:.4f}")
print(f"Significant difference: {sig_test['significant']}")

# Visualization and Reporting


### Create Comparison Visualizations

In [ ]:
def create_comparison_plots(comparison_df):
    """
    Generate comprehensive comparison visualizations
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Accuracy comparison
    axes[0, 0].bar(comparison_df['Model'], comparison_df['Overall_Accuracy'])
    axes[0, 0].set_title('Overall Accuracy Comparison')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].axhline(y=0.98, color='r', linestyle='--', label='Paper Baseline (98%)')
    axes[0, 0].legend()
    axes[0, 0].set_ylim([0.95, 1.0])
    
    # Single-node misclassification
    axes[0, 1].bar(
        comparison_df['Model'], 
        comparison_df['SingleNode_Misclassification'] * 100
    )
    axes[0, 1].set_title('Single-Node Task Misclassification Rate')
    axes[0, 1].set_ylabel('Misclassification (%)')
    axes[0, 1].axhline(y=1.5, color='r', linestyle='--', label='Paper Lower Bound')
    axes[0, 1].axhline(y=1.8, color='r', linestyle='--', label='Paper Upper Bound')
    axes[0, 1].legend()
    
    # F1-Score comparison
    axes[1, 0].bar(comparison_df['Model'], comparison_df['Overall_F1'])
    axes[1, 0].set_title('F1-Score Comparison')
    axes[1, 0].set_ylabel('F1-Score')
    axes[1, 0].set_ylim([0.95, 1.0])
    
    # Inference time
    axes[1, 1].bar(comparison_df['Model'], comparison_df['Inference_Time_per_Sample_ms'])
    axes[1, 1].set_title('Inference Time per Sample')
    axes[1, 1].set_ylabel('Time (ms)')
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=300)
    plt.show()

create_comparison_plots(comparison_df)

### Generate Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrices(models, X_test, y_test):
    """
    Plot confusion matrices for all models
    """
    n_models = len(models)
    fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    
    for idx, (name, model) in enumerate(models.items()):
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        
        # Normalize
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        sns.heatmap(
            cm_normalized, 
            annot=True, 
            fmt='.2f', 
            cmap='Blues',
            ax=axes[idx] if n_models > 1 else axes,
            cbar_kws={'label': 'Proportion'}
        )
        axes[idx].set_title(f'{name} Confusion Matrix')
        axes[idx].set_ylabel('True Label')
        axes[idx].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    plt.savefig('confusion_matrices.png', dpi=300)
    plt.show()

plot_confusion_matrices(models, X_test, y_test)

### Verify Checklist